# Hansen Ch.7 习题解答（计算部分）

理论见 `Hansen_Ch07_Exercises_Solutions.md`。

本 notebook：**Exercise 7.28**。

## Exercise 7.4 矩核对（数值）

In [ ]:
import numpy as np
# X1,X2 in {-1,1} with given joint probs
# outcomes: (1,1),(1,-1),(-1,1),(-1,-1) with probs 3/8,1/8,1/8,3/8
vals = np.array([[1,1],[1,-1],[-1,1],[-1,-1]], float)
p = np.array([3/8,1/8,1/8,3/8])
X1, X2 = vals[:,0], vals[:,1]
same = (X1==X2)
sig2 = np.where(same, 5/4, 1/4)

print('E[X1]=', np.sum(p*X1))
print('E[X1^2]=', np.sum(p*X1**2))
print('E[X1X2]=', np.sum(p*X1*X2))
print('E[e^2]=', np.sum(p*sig2))
print('E[X1^2 e^2]=', np.sum(p*(X1**2)*sig2))
print('E[X1X2 e^2]=', np.sum(p*X1*X2*sig2))
print('targets: 0,1,0.5,1,1,0.875')


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

CPS = Path("../../hansen/econometrics/data/cps09mar/cps09mar.xlsx")
if not CPS.exists():
    CPS = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/cps09mar/cps09mar.xlsx")
df = pd.read_excel(CPS)
df["experience"] = df["age"] - df["education"] - 6
df["lwage"] = np.log(df["earnings"]/(df["hours"]*df["week"]))
df["exp2"] = (df["experience"]**2)/100
s = df[(df.race==1)&(df.female==0)&(df.hisp==1)].copy()
y = s.lwage.to_numpy(float)
X = np.c_[s.education, s.experience, s.exp2, np.ones(len(s))]
names = ["education","experience","exp2/100","intercept"]
n,k = X.shape
beta = np.linalg.lstsq(X,y,rcond=None)[0]
e = y - X@beta
XXinv = np.linalg.inv(X.T@X)
h = np.sum(X*(X@XXinv), axis=1)
u = X * (e/(np.clip(1-h,1e-12,None)))[:,None]  # HC3
V = XXinv @ (u.T@u) @ XXinv
se = np.sqrt(np.diag(V))
print(pd.DataFrame({"beta":beta,"HC3_SE":se}, index=names))
print("n=", n)


### (b)–(d) $\theta=\beta_1/(\beta_2+0.2\beta_3)$（experience=10 时教育相对经验回报比）

In [ ]:
b1,b2,b3,b0 = beta
den = b2 + 0.2*b3
theta = b1/den
grad = np.array([1/den, -b1/den**2, -0.2*b1/den**2, 0.0])
se_theta = float(np.sqrt(grad @ V @ grad))
z90 = stats.norm.ppf(0.95)
print(f"theta = {theta:.4f}")
print(f"se(theta) = {se_theta:.4f}")
print(f"90% CI = [{theta - z90*se_theta:.4f}, {theta + z90*se_theta:.4f}]")


### (e) 回归函数在 education=12, experience=20

In [ ]:
x = np.array([12.0, 20.0, 4.0, 1.0])  # exp2/100 = 400/100=4
m = float(x @ beta)
se_m = float(np.sqrt(x @ V @ x))
z95 = 1.96
print(f"m(12,20) = {m:.4f}")
print(f"95% CI = [{m-z95*se_m:.4f}, {m+z95*se_m:.4f}]")


### (f) 样本外预测：edu=16, exp=5，80% 预测区间

In [ ]:
s2 = float(np.sum(e**2)/(n-k))
x = np.array([16.0, 5.0, 0.25, 1.0])
yhat = float(x @ beta)
se_f = float(np.sqrt(x @ V @ x + s2))
z80 = stats.norm.ppf(0.90)
lo, hi = yhat - z80*se_f, yhat + z80*se_f
print(f"point forecast log wage = {yhat:.4f}")
print(f"80% PI log wage = [{lo:.4f}, {hi:.4f}]")
print(f"80% PI wage = [{np.exp(lo):.2f}, {np.exp(hi):.2f}]")
